# Fine-tune Gemma 4 E4B for the mixer assistant

Teaches Gemma 4 to turn a producer's request ("pull the kick down 3 dB and add some reverb to the vocals") into tool calls for our Ableton-style app.
Based on Unsloth's official Gemma 4 notebook, with LoRA training on a free Colab GPU.

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. When the upload button appears (step 2), choose `train.jsonl` and `val.jsonl` from the project's `data/` folder.

| Step | What happens | Time (rough) |
|---|---|---|
| 1 | Install Unsloth | ~3 min |
| 2 | Upload the training data | ~1 min |
| 3–5 | Load Gemma 4 E4B and add the LoRA adapter | ~5 min |
| 6–7 | Turn our examples into Gemma's chat format and set up training | ~2 min |
| 8 | Train (`MAX_STEPS` below) | the long part: the cell prints the time per step |
| 9 | Try it on examples it has never seen | ~2 min |
| 10 | Save the adapter and download it | ~1 min |
| 11 | Optional: export a full GGUF model to Google Drive (~8–9 GB) | ~20 min |

## 1. Install Unsloth (from Unsloth's Gemma 4 notebook)

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
import torch; torch._dynamo.config.recompile_limit = 64;

## 2. Upload `train.jsonl` and `val.jsonl`

In [ ]:
import os
if not (os.path.exists("train.jsonl") and os.path.exists("val.jsonl")):
    from google.colab import files
    files.upload()   # pick train.jsonl and val.jsonl from the project's data/ folder
print({f: os.path.getsize(f) for f in ["train.jsonl", "val.jsonl"]})

## 3. Settings
* `MAX_STEPS`: each step trains on 4 examples. 300 steps = 1,200 examples. Raise it if you have time (750 steps = the whole training set once).
* `LOAD_IN_4BIT`: must stay `True` on the free T4 GPU (E4B in full precision needs ~17 GB).

In [ ]:
MODEL = "unsloth/gemma-4-E4B-it"
MAX_SEQ_LENGTH = 4096
MAX_STEPS = 300
LOAD_IN_4BIT = True

## 4. Load Gemma 4 E4B

In [ ]:
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = MODEL,
    dtype = None,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
    full_finetuning = False,
)

## 5. Add the LoRA adapter (text layers only)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

## 6. Turn our examples into Gemma's chat format
Each example is: the rules (system prompt) + the session and the request (user) + the correct tool calls (model), with the tool definitions.
We keep Gemma's **own** chat template, which knows how to write tools and tool calls, and turn thinking off (the app runs it with `think: false`).

In [ ]:
import json
from datasets import Dataset

text_tok = getattr(tokenizer, "tokenizer", tokenizer)   # the plain text tokenizer
def render(ex, generation_prompt = False):
    msgs = ex["messages"] if not generation_prompt else ex["messages"][:2]
    return text_tok.apply_chat_template(msgs, tools = ex["tools"], tokenize = False,
                                         add_generation_prompt = generation_prompt, enable_thinking = False)

load = lambda path: [json.loads(line) for line in open(path)]
train_rows, val_rows = load("train.jsonl"), load("val.jsonl")
texts = [render(ex).removeprefix("<bos>") for ex in train_rows]
lengths = [len(text_tok(t)["input_ids"]) for t in texts]
keep = [t for t, n in zip(texts, lengths) if n <= MAX_SEQ_LENGTH]
print(f"{len(texts)} examples, tokens per example: min {min(lengths)}, mean {sum(lengths)//len(lengths)}, max {max(lengths)}; kept {len(keep)} (<= {MAX_SEQ_LENGTH})")
dataset = Dataset.from_dict({"text": keep})
print(texts[0][-500:])   # the end of one example: the correct tool calls

## 7. Set up training (only the model's answer is learned)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = MAX_STEPS,
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)
trainer = train_on_responses_only(trainer, instruction_part = "<|turn>user\n", response_part = "<|turn>model\n")
# Check: only the tool calls should remain below (everything else is masked out of the loss)
labels = trainer.train_dataset[0]["labels"]
print(text_tok.decode([x for x in labels if x != -100]))

## 8. Train  (loss should drop and level off; Unsloth says E4B losses look high at first, which is normal)

In [ ]:
gpu = torch.cuda.get_device_properties(0)
print(f"GPU = {gpu.name}, {round(gpu.total_memory / 1024**3, 1)} GB")
stats = trainer.train()
print(f"Trained {MAX_STEPS} steps in {round(stats.metrics['train_runtime'] / 60, 1)} minutes; peak memory {round(torch.cuda.max_memory_reserved() / 1024**3, 1)} GB")

## 9. Try it on validation examples it has never seen

In [ ]:
for ex in val_rows[:5]:
    inputs = text_tok.apply_chat_template(ex["messages"][:2], tools = ex["tools"], add_generation_prompt = True, enable_thinking = False,
                                           tokenize = True, return_dict = True, return_tensors = "pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens = 400, do_sample = False)
    print("REQUEST :", ex["messages"][1]["content"].split("Request: ")[-1])
    print("MODEL   :", text_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens = False))
    print("EXPECTED:", [(c["function"]["name"], c["function"]["arguments"]) for c in ex["messages"][2]["tool_calls"]])
    print()

## 10. Save the LoRA adapter (small) and download it

In [ ]:
model.save_pretrained("gemma4_mixer_lora")
tokenizer.save_pretrained("gemma4_mixer_lora")
!zip -qr gemma4_mixer_lora.zip gemma4_mixer_lora
from google.colab import files
files.download("gemma4_mixer_lora.zip")

## 11. Optional: export the full model as a GGUF file to Google Drive
Ollama can load this directly. For Gemma 4, Unsloth currently exports Q8_0, BF16 or F16 only; Q8_0 of E4B is about 8–9 GB.
This merges the adapter into the model, so it needs time and memory; if Colab runs out of memory here, the adapter from step 10 is still saved.

In [ ]:
EXPORT_GGUF = True
if EXPORT_GGUF:
    from google.colab import drive
    drive.mount("/content/drive")
    model.save_pretrained_gguf("/content/drive/MyDrive/gemma4_mixer", tokenizer, quantization_method = "Q8_0")
    !ls -lh /content/drive/MyDrive/gemma4_mixer